# IPL Winner Prediction

Simple notebook using only these features:
- team1
- team2
- venue
- toss_decision
- first_innings_score

Target column: `winner`

Models used:
- Decision Tree
- Random Forest

In [19]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [20]:
df = pd.read_csv('ipl.csv')

team_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

active_teams = [
    'Chennai Super Kings',
    'Delhi Capitals',
    'Gujarat Titans',
    'Kolkata Knight Riders',
    'Lucknow Super Giants',
    'Mumbai Indians',
    'Punjab Kings',
    'Rajasthan Royals',
    'Royal Challengers Bengaluru',
    'Sunrisers Hyderabad'
]

for col in ['team1', 'team2', 'winner']:
    df[col] = df[col].replace(team_map)

df['venue'] = df['venue'].fillna('Unknown Venue')
df['toss_decision'] = df['toss_decision'].fillna('Unknown')
df['target_runs'] = pd.to_numeric(df['target_runs'], errors='coerce')

df = df.dropna(subset=['team1', 'team2', 'venue', 'toss_decision', 'target_runs', 'winner']).copy()

df = df[
    df['team1'].isin(active_teams)
    & df['team2'].isin(active_teams)
    & df['winner'].isin(active_teams)
].copy()

# In IPL data, target_runs is usually first innings score + 1.\n
df['first_innings_score'] = df['target_runs'] - 1

df = df[['team1', 'team2', 'venue', 'toss_decision', 'first_innings_score', 'winner']].copy()
df.head()

,team1,team2,venue,toss_decision,first_innings_score,winner
0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,field,222.0,Kolkata Knight Riders
1,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",bat,240.0,Chennai Super Kings
2,Delhi Capitals,Rajasthan Royals,Feroz Shah Kotla,bat,129.0,Delhi Capitals
3,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,bat,165.0,Royal Challengers Bengaluru
5,Rajasthan Royals,Punjab Kings,Sawai Mansingh Stadium,bat,166.0,Rajasthan Royals


In [21]:
X = df[['team1', 'team2', 'venue', 'toss_decision', 'first_innings_score']]
y = df['winner']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = ['team1', 'team2', 'venue', 'toss_decision']
numeric_features = ['first_innings_score']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=4, random_state=42)
}

In [22]:
results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    results.append({'Model': name, 'Accuracy': accuracy})
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
results_df

,Model,Accuracy
1,Random Forest,0.544041
0,Decision Tree,0.430052


In [23]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print('Best Model:', best_model_name)
print('Accuracy:', round(accuracy_score(y_test, best_predictions), 4))
print()
print(classification_report(y_test, best_predictions, zero_division=0))

Best Model: Random Forest
Accuracy: 0.544

                             precision    recall  f1-score   support

        Chennai Super Kings       0.62      0.81      0.70        26
             Delhi Capitals       0.36      0.24      0.29        21
             Gujarat Titans       0.83      0.71      0.77         7
      Kolkata Knight Riders       0.53      0.71      0.61        24
       Lucknow Super Giants       1.00      0.17      0.29         6
             Mumbai Indians       0.59      0.93      0.72        28
               Punjab Kings       0.67      0.10      0.17        21
           Rajasthan Royals       0.42      0.40      0.41        20
Royal Challengers Bengaluru       0.46      0.57      0.51        23
        Sunrisers Hyderabad       0.58      0.41      0.48        17

                   accuracy                           0.54       193
                  macro avg       0.61      0.50      0.49       193
               weighted avg       0.56      0.54      0.51

In [24]:
sample_match = pd.DataFrame([{
    'team1': 'Chennai Super Kings',
    'team2': 'Mumbai Indians',
    'venue': 'Wankhede Stadium',
    'toss_decision': 'field',
    'first_innings_score': 180
}])

predicted_winner = best_model.predict(sample_match)[0]
print('Predicted Winner:', predicted_winner)

Predicted Winner: Mumbai Indians


In [26]:
import joblib

joblib.dump(best_model, "IPL_Winner_Prediction_Clean.pkl")

['IPL_Winner_Prediction_Clean.pkl']